# policy-engine deep dive

> Master kernel policies, evaluation, and audit on `policy-engine`.

This notebook is a port of the agent_os tutorial `06-policy-engine.ipynb`,
rebuilt against this repo's `policy-engine`. The engine has a deliberately
small surface — `BaseKernel`, `GovernancePolicy`, `PolicyRequest`,
`PolicyDecision`, `audit()`/`AUDIT`, `PolicyViolationError`,
`ExecutionContext`. Where the original notebook reaches for primitives that
do not exist here (regex `Pattern`, `Rule`/`Condition`, `AgentSignal`, YAML
loader, `PolicyDebugger`, named templates), we call out the gap and show
how the same idea is expressed with what the engine actually ships.

## Learning Objectives

By the end of this notebook, you will:

1. Understand `BaseKernel.evaluate` as the single gate.
2. Compose a `GovernancePolicy` from patterns, tools, the max-call cap,
   and the human-approval flag.
3. Map "rules" to the engine's fixed evaluation order.
4. Express SIGKILL / SIGSTOP / SIGCONT as host-side responses to a
   `PolicyDecision`.
5. Inspect `AUDIT` and verify raw payloads are never stored.

---


## The Core Idea: Kernel vs. Prompt Safety

    Prompt-Based Safety:          Kernel-Based Safety:

      "Please don't do X"           Every action is checked
             v                             v
      LLM decides to comply         Kernel decides to allow
             v                             v
      Maybe it doesn't              No choice -- blocked or allowed

In this engine, the **kernel** is `BaseKernel.evaluate(ctx, request)`. It
returns a structured `PolicyDecision`; the host decides what to do when
`allowed=False`.

---


## Step 1: Install dependencies


In [ ]:
!pip install -e ../policy-engine --quiet


In [ ]:
"""Bootstrap: add policy-engine src to sys.path, import primitives + step()."""

import sys
from pathlib import Path

# When run via the shim, __notebook_dir__ is injected into globals.
# When run interactively in Jupyter, fall back to cwd.
HERE = Path(globals().get("__notebook_dir__") or Path.cwd()).resolve()
REPO_ROOT = HERE.parent

for _src in (
    REPO_ROOT / "policy-engine" / "src",
    REPO_ROOT.parent / "packages" / "policy-engine" / "src",
):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))
        break

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from policy_engine import (  # noqa: E402
    AUDIT,
    BaseKernel,
    ExecutionContext,
    GovernancePolicy,
    PolicyRequest,
    PolicyViolationError,
    audit,
)
from _shared import reset_steps, step  # noqa: E402

FRAMEWORK = "policy_deep_dive"
reset_steps(FRAMEWORK)
step(FRAMEWORK, "imports + sys.path bootstrap complete")


## Step 2: Policy "templates"

The original notebook calls `KernelSpace(policy="strict")`. `policy-engine`
has no template registry: a `GovernancePolicy` is a plain dataclass, so
"templates" are just factory functions. Below are three reference shapes
mirroring the original (`strict`, `permissive`, `audit`).


In [ ]:
def strict_policy() -> GovernancePolicy:
    """Many blocked patterns + tools, low call cap, requires approval."""
    return GovernancePolicy(
        name="strict",
        blocked_patterns=[
            "DROP TABLE", "rm -rf", "ignore previous instructions",
            "reveal system prompt", "<system>", "ssn", "password",
        ],
        blocked_tools=["shell_exec", "network_request", "file_write", "file_delete"],
        max_tool_calls=3,
        require_human_approval=True,
    )


def permissive_policy() -> GovernancePolicy:
    """Empty blocks, generous cap. Useful for trusted, sandboxed flows."""
    return GovernancePolicy(
        name="permissive",
        blocked_patterns=[],
        max_tool_calls=100,
    )


def audit_policy() -> GovernancePolicy:
    """No blocks, but every call passes through evaluate() so AUDIT records it."""
    return GovernancePolicy(name="audit", max_tool_calls=1000)


step(FRAMEWORK, "defined three policy templates")

print("Policy comparison")
print("=" * 60)
for p in (strict_policy(), permissive_policy(), audit_policy()):
    print(f"\n  name: {p.name}")
    print(f"    blocked_patterns: {len(p.blocked_patterns)}")
    print(f"    blocked_tools:    {p.blocked_tools or []}")
    print(f"    max_tool_calls:   {p.max_tool_calls}")
    print(f"    require_human_approval: {p.require_human_approval}")


## Step 3: Custom policy

Vocabulary differences vs. the original:

- **`allowed_actions` / `blocked_actions` -> `allowed_tools` / `blocked_tools`.**
  The engine speaks "tools," not "actions."
- **regex `Pattern(r"\bssn\b", ...)` -> plain substrings.**
  `GovernancePolicy.matches_pattern` is **case-insensitive substring**
  containment only. Translate `\bssn\b` -> `"ssn"`,
  `DROP\s+TABLE` -> `"DROP TABLE"`, `\bcredit.?card\b` -> `"credit card"`.
- **`on_violation: SIGKILL` -> the engine returns `PolicyDecision`.**
  How the host responds (raise, queue for approval, log) is its concern.


In [ ]:
analyst_policy = GovernancePolicy(
    name="data-analyst",
    blocked_tools=[
        "write_file", "delete_file", "send_email",
        "execute_shell", "network_request",
    ],
    blocked_patterns=[
        "ssn",
        "credit card",
        "DROP TABLE",
        "password",
    ],
    max_tool_calls=20,
)

step(FRAMEWORK, "constructed analyst_policy")
print("Custom policy")
print(f"  name:             {analyst_policy.name}")
print(f"  blocked_tools:    {analyst_policy.blocked_tools}")
print(f"  blocked_patterns: {analyst_policy.blocked_patterns}")
print(f"  max_tool_calls:   {analyst_policy.max_tool_calls}")


In [ ]:
kernel = BaseKernel(analyst_policy)
ctx = kernel.create_context("analyst")

ok = kernel.evaluate(ctx, PolicyRequest(payload="Show Q4 revenue trends"))
bad = kernel.evaluate(ctx, PolicyRequest(payload="Get user ssn from logs"))

step(FRAMEWORK, f"benign request -> allowed={ok.allowed}")
step(FRAMEWORK, f"PII request    -> allowed={bad.allowed} reason={bad.reason}")

print("Allowed decision:")
print(f"  allowed={ok.allowed}  policy={ok.policy}  payload_hash={ok.payload_hash[:12]}...")
print("Blocked decision:")
print(f"  allowed={bad.allowed}  reason={bad.reason}  matched_pattern={bad.matched_pattern}")
print(f"\nctx.call_count after one allowed + one blocked: {ctx.call_count}")
print("(blocked calls do not increment the counter)")


## Step 4: Rules and conditions

The original notebook builds `Rule(condition=lambda ctx: ..., action=BLOCK)`
with arbitrary lambda checks. The engine deliberately does not ship that —
"rules" here are encoded as **fields** on `GovernancePolicy`, evaluated in a
**fixed order** by `BaseKernel.evaluate`:

1. `max_tool_calls` cap (`ctx.call_count >= policy.max_tool_calls`)
2. `blocked_tools` / `allowed_tools` (when `request.tool_name` is set)
3. `require_human_approval`
4. `blocked_patterns` against `request.payload`

Each blocked decision surfaces a stable `reason` string
(`max_tool_calls exceeded`, `blocked_tool:<name>`, `tool_not_allowed:<name>`,
`human_approval_required`, `blocked_pattern:<pattern>`) plus the matched
pattern or tool name. For richer conditional logic you compose multiple
kernels or wrap `evaluate` in your host code.


In [ ]:
# (a) Tool allow/deny
tool_policy = GovernancePolicy(
    name="tool-gated",
    blocked_tools=["network_request"],
    max_tool_calls=10,
)
tool_kernel = BaseKernel(tool_policy)
tool_ctx = tool_kernel.create_context("tool")

dec = tool_kernel.evaluate(
    tool_ctx,
    PolicyRequest(payload="fetch external resource", tool_name="network_request"),
)
print(f"(a) tool allow/deny    allowed={dec.allowed} reason={dec.reason} tool={dec.tool_name}")

# (b) Rate limit ~ max_tool_calls
rate_policy = GovernancePolicy(name="rate-limited", max_tool_calls=10)
rate_kernel = BaseKernel(rate_policy)
rate_ctx = rate_kernel.create_context("rate")

allowed_count = 0
for i in range(11):
    d = rate_kernel.evaluate(rate_ctx, PolicyRequest(payload=f"call {i}"))
    if d.allowed:
        allowed_count += 1
print(f"(b) rate limit        allowed_count={allowed_count}/11  ctx.call_count={rate_ctx.call_count}")
print(f"    final decision    allowed={d.allowed} reason={d.reason}")

# (c) Human approval gate
approval_policy = GovernancePolicy(name="approval", require_human_approval=True)
approval_kernel = BaseKernel(approval_policy)
approval_ctx = approval_kernel.create_context("approval")

dec = approval_kernel.evaluate(approval_ctx, PolicyRequest(payload="contact external API"))
print(f"(c) human approval     allowed={dec.allowed} requires_approval={dec.requires_approval} reason={dec.reason}")

step(FRAMEWORK, "exercised three rule shapes via policy fields")


## Step 5: Responding to a `PolicyDecision`

The original notebook ships a signal pub/sub
(`@kernel.on_signal(AgentSignal.SIGKILL)`). `policy-engine` does not — it
returns a structured `PolicyDecision` and gets out of the way. Three host
response styles cover the same ground:

- **SIGKILL** -- raise `PolicyViolationError` and let the call stack unwind.
- **SIGSTOP / SIGCONT** -- treat `decision.requires_approval=True` as a
  pause; resume after a (mocked) human ack.
- **audit-only** -- log the decision and continue.


In [ ]:
policy = strict_policy()
kernel = BaseKernel(policy)
ctx = kernel.create_context("response-styles")

# (a) SIGKILL analogue: raise on block.
def kill_on_block(decision):
    if not decision.allowed:
        audit(FRAMEWORK, "evaluate", "BLOCKED", "raise-on-block", decision=decision)
        raise PolicyViolationError(decision.reason or "blocked",
                                    pattern=decision.matched_pattern)
    audit(FRAMEWORK, "evaluate", "ALLOWED", "raise-on-block", decision=decision)

try:
    kill_on_block(kernel.evaluate(ctx, PolicyRequest(payload="please DROP TABLE users")))
except PolicyViolationError as exc:
    print(f"(a) SIGKILL ~ raised: {exc.reason} (pattern={exc.pattern})")

# (b) SIGSTOP/SIGCONT analogue: queue requests for approval.
pending_review = []

def pause_on_approval(decision, request):
    if decision.requires_approval:
        pending_review.append(request)
        audit(FRAMEWORK, "evaluate", "PAUSED", "queue-for-approval", decision=decision)
        print(f"(b) SIGSTOP ~ queued for human review: {decision.reason}")
        # mock human approval
        approved = pending_review.pop()
        audit(FRAMEWORK, "approval", "APPROVED", "human-ack", decision=decision)
        print(f"(b) SIGCONT ~ approved by mock-reviewer; resuming: {approved.payload_sha256()[:12]}...")
        return True
    return decision.allowed

req = PolicyRequest(payload="contact external API")
pause_on_approval(kernel.evaluate(ctx, req), req)

# (c) audit-only: just record.
audit_dec = kernel.evaluate(ctx, PolicyRequest(payload="benign read"))
audit(FRAMEWORK, "evaluate", "ALLOWED" if audit_dec.allowed else "BLOCKED",
      "audit-only", decision=audit_dec)
print(f"(c) audit-only       allowed={audit_dec.allowed}")

step(FRAMEWORK, "demonstrated three response styles")


## Step 6: Policy files

The engine stays stdlib-only — no YAML loader. Persistence and config
parsing are host concerns. Below is a JSON round-trip using the stdlib;
PyYAML would be a one-liner if you have it installed.


In [ ]:
import json
from dataclasses import asdict

source = analyst_policy
path = HERE / "security.json"
path.write_text(json.dumps(asdict(source), indent=2))
print(f"wrote {path.name} ({path.stat().st_size} bytes)")

loaded = GovernancePolicy(**json.loads(path.read_text()))
loaded_kernel = BaseKernel(loaded)
loaded_ctx = loaded_kernel.create_context("loaded")
dec = loaded_kernel.evaluate(loaded_ctx, PolicyRequest(payload="show last quarter results"))

print(f"loaded.name      = {loaded.name}")
print(f"loaded.blocked_patterns = {loaded.blocked_patterns}")
print(f"round-tripped decision: allowed={dec.allowed}  policy={dec.policy}")

step(FRAMEWORK, "JSON round-trip via dataclasses.asdict")


## Step 7: Policy debugging

`kernel.evaluate(ctx, request)` already returns a structured
`PolicyDecision`, so it *is* the debugger. No separate `PolicyDebugger`
class is needed — and none exists.

The substring-only matcher is the one footgun to watch for: a regex-style
pattern like `\bssn\b` would never match (it would search for the literal
backslash characters). The Step 7 table shows that translated substrings
work, while a regex-only pattern silently does not.


In [ ]:
debug_kernel = BaseKernel(analyst_policy)
debug_ctx = debug_kernel.create_context("debug")

cases = [
    PolicyRequest(payload="read /data/sales.csv",            tool_name="read_file"),
    PolicyRequest(payload="write to /data/output.csv",       tool_name="write_file"),
    PolicyRequest(payload="SELECT name FROM users",          tool_name="query_database"),
    PolicyRequest(payload="SELECT ssn FROM users",           tool_name="query_database"),
]

print(f"{'tool':18} {'allowed':8} {'reason':28} {'matched':12}")
print("-" * 72)
for req in cases:
    d = debug_kernel.evaluate(debug_ctx, req)
    print(f"{req.tool_name or '-':18} {str(d.allowed):8} "
          f"{(d.reason or '-'):28} {(d.matched_pattern or '-'):12}")

# Honesty check: a regex-style "pattern" passed as a substring will NOT match.
honesty = GovernancePolicy(name="regex-trap", blocked_patterns=[r"\bssn\b"])
honesty_kernel = BaseKernel(honesty)
honesty_ctx = honesty_kernel.create_context("honesty")
trap = honesty_kernel.evaluate(honesty_ctx, PolicyRequest(payload="leak ssn now"))
print()
print(f"regex-style pattern \\bssn\\b on payload 'leak ssn now' -> "
      f"allowed={trap.allowed} (expected True; substring is the literal '\\bssn\\b')")

step(FRAMEWORK, "evaluated debug table + regex-trap honesty check")


## Step 8: Audit trail

`policy-engine` ships an in-memory `AUDIT: list[dict]` and a single helper
`audit(framework, phase, status, detail, *, decision=...)`. Each entry
carries `ts`, `framework`, `phase`, `status`, `detail`, plus optional
`policy`, `reason`, `tool_name`, and `payload_hash`. **Raw payload is never
stored** -- only the SHA-256 hash. Persistence (rotating files, syslog,
OTel) is a host concern.


In [ ]:
start = len(AUDIT)
walk_kernel = BaseKernel(analyst_policy)
walk_ctx = walk_kernel.create_context("walk")

for payload in [
    "summarize Q4 revenue",
    "include user ssn for personalization",
    "DROP TABLE customers",
]:
    d = walk_kernel.evaluate(walk_ctx, PolicyRequest(payload=payload))
    audit(FRAMEWORK, "evaluate", "ALLOWED" if d.allowed else "BLOCKED",
          f"audit-walkthrough", decision=d)

new_entries = AUDIT[start:]
print(f"new audit entries: {len(new_entries)}")
print("-" * 72)
for entry in new_entries:
    print(f"  ts={entry['ts']}")
    print(f"    framework={entry['framework']} phase={entry['phase']} status={entry['status']}")
    print(f"    detail={entry['detail']}")
    if entry.get("reason"):
        print(f"    reason={entry['reason']}")
    if entry.get("payload_hash"):
        print(f"    payload_hash={entry['payload_hash'][:16]}...  (raw payload absent by design)")
    print()

step(FRAMEWORK, f"appended {len(new_entries)} audit entries")


## Cleanup


In [ ]:
path = HERE / "security.json"
if path.exists():
    path.unlink()
    print(f"removed {path.name}")
else:
    print("nothing to remove")

step(FRAMEWORK, "cleanup complete")


---

## Summary

| Engine primitive | What it does |
|---|---|
| `GovernancePolicy` | Dataclass: `blocked_patterns`, `blocked_tools`, `allowed_tools`, `max_tool_calls`, `require_human_approval`. |
| `BaseKernel.evaluate(ctx, request)` | The single gate. Returns `PolicyDecision`. Increments `ctx.call_count` only when `allowed=True`. |
| `PolicyRequest(payload, tool_name, phase)` | Frozen request. `payload_sha256()` is used by audit; raw payload is never stored. |
| `PolicyDecision` | `allowed`, `reason`, `policy`, `matched_pattern`, `tool_name`, `requires_approval`, `payload_hash`, `phase`. |
| `audit(framework, phase, status, detail, *, decision=...)` | Append a record to `AUDIT`. |
| `PolicyViolationError(reason, pattern=...)` | Raise from host code when the policy denies. |
| `ExecutionContext` | Per-run state (`name`, `policy`, `call_count`). |

### Quick reference

```python
from policy_engine import (
    AUDIT, BaseKernel, ExecutionContext, GovernancePolicy,
    PolicyRequest, PolicyViolationError, audit,
)

policy = GovernancePolicy(
    name="custom",
    blocked_patterns=["DROP TABLE", "ssn"],
    blocked_tools=["network_request"],
    max_tool_calls=10,
    require_human_approval=False,
)
kernel = BaseKernel(policy)
ctx = kernel.create_context("agent-1")

decision = kernel.evaluate(ctx, PolicyRequest(payload="Show revenue"))
if not decision.allowed:
    raise PolicyViolationError(decision.reason, pattern=decision.matched_pattern)

audit("my-framework", "evaluate", "ALLOWED" if decision.allowed else "BLOCKED",
      decision=decision)
```

---

## Next steps

- `governance_showcase.py` -- the same primitives composed across a
  multi-stage agent flow.
- The seven framework adapters under `policy_engine.adapters` -- each
  plugs the kernel into a different SDK.
- `claude_all_hooks.py` -- every Python-supported Claude Agent SDK hook
  factory wired to one shared kernel.
